# 05. How predictable is yield?

Notebook 04 asked how much fertilizer matters. This asks something different: can yield
be predicted at all from these variables?

The answer turns out to be mostly no, and I report that rather than hiding it behind a
weak comparison.

**Output:** `ml_results.json`

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

from src import config as C

d = pd.read_csv(C.PROCESSED / "model_frame.csv").sort_values(["iso3", "year"]).reset_index(drop=True)

FEATURES = ["ln_fert", "ln_area", "rural_pop_pct", "ln_pop", "year"]
d = d.dropna(subset=FEATURES + ["ln_yield"]).reset_index(drop=True)

print("rows:", len(d), "| countries:", d["iso3"].nunique())

rows: 9466 | countries: 178


## 1. Split on time, not at random

A random split would put Nigeria 2015 in training and Nigeria 2016 in testing. Yields
barely move in a year, so the model would already have seen the answer.

Splitting on time matches how it would actually be used: learn from the past, predict
forward.

In [2]:
train = d[d["year"] <  C.ML_SPLIT_YEAR]
test  = d[d["year"] >= C.ML_SPLIT_YEAR]

print(f"train: {len(train):>6,}  {train['year'].min()}-{train['year'].max()}")
print(f"test:  {len(test):>6,}  {test['year'].min()}-{test['year'].max()}")

train:  7,385  1961-2012
test:   2,081  2013-2024


In [3]:
def fit_score(tr, te, feats, label):
    m = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06,
                                      max_depth=6, random_state=C.RANDOM_SEED)
    m.fit(tr[feats], tr["ln_yield"])
    pred = m.predict(te[feats])
    r2  = r2_score(te["ln_yield"], pred)
    mae = mean_absolute_error(te["ln_yield"], pred)
    print(f"{label:<32} R2={r2:7.4f}  typical error ~{(np.exp(mae) - 1) * 100:.0f}%")
    return m, r2, mae

model, r2_time, _ = fit_score(train, test, FEATURES, "time split")

rtr, rte = train_test_split(d, test_size=len(test) / len(d), random_state=C.RANDOM_SEED)
_, r2_random, _ = fit_score(rtr, rte, FEATURES, "random split (leaks)")

print(f"\nthe random split looks {r2_random - r2_time:.3f} better than it is")

time split                       R2= 0.7206  typical error ~35%


random split (leaks)             R2= 0.8677  typical error ~21%

the random split looks 0.147 better than it is


## 2. Compare against something simple

An R squared on its own means nothing. It has to beat a simple rule, and the simple rule
should be the best one I can think of, not the weakest.

In [4]:
last_known = train.sort_values("year").groupby("iso3")["ln_yield"].last()

baselines = {
    "one average for everyone":
        np.full(len(test), train["ln_yield"].mean()),
    "each country's own average":
        test["iso3"].map(train.groupby("iso3")["ln_yield"].mean())
            .fillna(train["ln_yield"].mean()).values,
    "each country's last known yield":
        test["iso3"].map(last_known).fillna(train["ln_yield"].mean()).values,
}

for name, pred in baselines.items():
    print(f"{name:<34} R2={r2_score(test['ln_yield'], pred):7.4f}")
print(f"{'my model':<34} R2={r2_time:7.4f}")

one average for everyone           R2=-0.3003
each country's own average         R2= 0.4926
each country's last known yield    R2= 0.8823
my model                           R2= 0.7206


The simplest useful rule beats my model. Assuming each country stays where it was
scores higher than 400 boosted trees.

If I had only used the weaker baselines the model would have looked like a success. That
is the whole reason to pick a strong one.

## 3. Work out why, then fix it

The baseline knows something my model does not: which country each row belongs to, and
where it already sits. None of my features say that.

So I give the model the same information. For the test rows I use the last value from
before the split, not the real previous year, so nothing from the future leaks in.

In [5]:
d["lag_yield"] = d.groupby("iso3")["ln_yield"].shift(1)

train2 = d[d["year"] <  C.ML_SPLIT_YEAR].dropna(subset=["lag_yield"]).copy()
test2  = d[d["year"] >= C.ML_SPLIT_YEAR].copy()
test2["lag_yield"] = test2["iso3"].map(last_known)
test2 = test2.dropna(subset=["lag_yield"])

FEATURES2 = FEATURES + ["lag_yield"]

r2_base  = r2_score(test2["ln_yield"], test2["lag_yield"])
mae_base = mean_absolute_error(test2["ln_yield"], test2["lag_yield"])
print(f"{'baseline, last known yield':<32} R2={r2_base:7.4f}")

_, r2_nolag, _ = fit_score(train2, test2, FEATURES,  "model without the lag")
model2, r2_lag, mae_lag = fit_score(train2, test2, FEATURES2, "model with the lag")

print(f"\nthe model now beats the baseline by {r2_lag - r2_base:+.4f}")

baseline, last known yield       R2= 0.8823


model without the lag            R2= 0.7209  typical error ~35%


model with the lag               R2= 0.8867  typical error ~20%

the model now beats the baseline by +0.0044


It wins by half a percentage point, for 400 trees. That is the honest size of it.

## 4. What the model is leaning on

In [6]:
def importance(m, te, feats):
    pi = permutation_importance(m, te[feats], te["ln_yield"], n_repeats=8,
                                random_state=C.RANDOM_SEED, scoring="r2")
    return pd.Series(pi.importances_mean, index=feats)

pd.DataFrame({
    "without lag": importance(model, test, FEATURES),
    "with lag":    importance(model2, test2, FEATURES2),
}).fillna(0).sort_values("with lag", ascending=False).round(4)

,without lag,with lag
lag_yield,0.0000,1.3281
ln_fert,0.7972,0.0300
ln_area,0.2844,0.0148
ln_pop,0.1223,0.0087
rural_pop_pct,0.1699,0.0054
year,0.0000,0.0000


Fertilizer looks like the most important thing until I add the lag, then it drops to
almost nothing.

Nothing about fertilizer changed. Last year's yield already contains it.

So importance tells me what the model is using given everything else it has. It does not
tell me what matters in the world. The regression in notebook 04 is the one that can
speak to that.

## 5. What I take from this

In [7]:
print(f"best simple rule:   R2 = {r2_base:.4f}")
print(f"my model:           R2 = {r2_lag:.4f}")
print(f"difference:              {r2_lag - r2_base:+.4f}")
print(f"typical error:      ~{(np.exp(mae_lag) - 1) * 100:.0f}%")

best simple rule:   R2 = 0.8823
my model:           R2 = 0.8867
difference:              +0.0044
typical error:      ~20%


Yields barely move from year to year, and most of what the model learns is that.
Beyond knowing where a country already is, these variables add very little.

That is worth knowing. It points at what better data would look like: soil, weather, seed
variety, farm level records. Not national averages.

It also sets out which model does which job. Notebook 04 carries the finding. This one
shows how far the data can be pushed.

In [8]:
ml_results = {
    "task": "predict ln(cereal yield) by country and year",
    "split": {"type": "time", "train_before": C.ML_SPLIT_YEAR,
              "n_train": int(len(train2)), "n_test": int(len(test2))},
    "model": "HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06, max_depth=6)",
    "r2_time_split_no_lag": round(float(r2_nolag), 4),
    "r2_random_split_no_lag": round(float(r2_random), 4),
    "r2_best_baseline": round(float(r2_base), 4),
    "r2_model_with_lag": round(float(r2_lag), 4),
    "improvement_over_baseline": round(float(r2_lag - r2_base), 4),
    "typical_error_pct": round(float((np.exp(mae_lag) - 1) * 100), 1),
    "baseline": "each country held at its last yield before the split",
    "takeaway": ("The model barely beats assuming nothing changes. Yields are very "
                 "persistent and these variables add little on top."),
}

with open(C.PROCESSED / "ml_results.json", "w") as f:
    json.dump(ml_results, f, indent=2)

print(json.dumps(ml_results, indent=2))

{
  "task": "predict ln(cereal yield) by country and year",
  "split": {
    "type": "time",
    "train_before": 2013,
    "n_train": 7207,
    "n_test": 2081
  },
  "model": "HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06, max_depth=6)",
  "r2_time_split_no_lag": 0.7209,
  "r2_random_split_no_lag": 0.8677,
  "r2_best_baseline": 0.8823,
  "r2_model_with_lag": 0.8867,
  "improvement_over_baseline": 0.0044,
  "typical_error_pct": 19.5,
  "baseline": "each country held at its last yield before the split",
  "takeaway": "The model barely beats assuming nothing changes. Yields are very persistent and these variables add little on top."
}


---

**Next:** `06_figures.ipynb`.